# LPG 原料価格と損益分岐点の分析

## このノートで分かること

main.py のベイズ最適化(BO)が見つけた**最適プロセス設計を一切変えずに**、
LPG(原料)の単価だけを変えたとき、年間利益(Profit)がどう動き、
**何円/kg を下回ると黒字になるか(損益分岐価格)** を求める。

## なぜ HYSYS を1回しか回さなくてよいか

「設計を固定したまま原料単価だけ変える」とき、変わるのは**お金の計算だけ**で、
プロセスの物理(各流れの流量・温度・圧力、必要熱量、装置サイズ、CAPEX)は**まったく変化しない**。
したがって:

1. 最適設計を **HYSYS で1回だけ**フル評価して、物理結果(`one_pass`)と経済内訳を得る。
2. 価格スイープは、その物理結果を固定したまま経済計算
   (`flowsheet.economics.calculate_economics`)の**単価定数だけを差し替えて**再計算する
   → HYSYS 不要・一瞬。

この「原料単価を変えても原料費の項以外は1円も動かない」ことは、後段(セル4)で
**実際にコードを走らせて実証**する(思い込みで済ませない)。

> ⚠️ HYSYS を使う**セル3だけ**は、京大ライセンスサーバへの **VPN 接続が必要**。
> それ以降のセルは HYSYS を使わないので VPN なしでも動く。

> 図の様式: グリッド線なし・四辺に内向き目盛り・配色はグレースケール+ハッチング
> (`Z:\report_for_processdesign\rule.md` の図表ルールに準拠)。


In [ ]:
# セル1: セットアップ(ライブラリ読み込み + 図の様式設定)
import os, sys, json
import dataclasses as dc
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- プロジェクト直下を import パスに通す ---
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# --- 日本語フォント(Windows 標準を順に探索)---
for _f in ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Hiragino Sans', 'Noto Sans CJK JP']:
    if any(_f.lower() == _ent.name.lower() for _ent in matplotlib.font_manager.fontManager.ttflist):
        matplotlib.rcParams['font.family'] = _f
        print(f"日本語フォント: {_f}")
        break
else:
    print("日本語フォントが見つからないため既定フォントを使用(軸ラベルが□になる場合あり)")

# --- レポート図表ルール (rule.md) に沿った様式 ---
matplotlib.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
matplotlib.rcParams['figure.dpi']    = 110
matplotlib.rcParams['axes.grid']     = False         # グリッド線は使わない
matplotlib.rcParams['xtick.direction'] = 'in'        # 目盛りは内向き
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['xtick.top']     = True          # 四辺すべてに目盛り
matplotlib.rcParams['ytick.right']   = True
matplotlib.rcParams['font.size']     = 11

# --- プロジェクトのモジュール(最適化器 optuna には依存しない構成)---
from config.load import load_operating_config
from flowsheet import FlowsheetDesignVars, evaluate
import flowsheet.economics as econ_mod       # ← この中の単価定数を差し替える
from flowsheet.economics import calculate_economics
from src.distillation_core import ColumnTunables
from units.reactors.catofin import CatofinDesignVars
from units.separators.psa.psa_system import PSADesignVars
from units.separators.membrane.membrane_system import MemDesignVars
from src.component_data import MW
from src import cost_parameters as cp

# --- 評価条件(main.py と同一)---
REACTOR_KIND = 'catofin'
P_L_Pa       = 1.0e5
APPLY_HI, APPLY_STAGE2, HI_DT_MIN_K = True, False, 10.0
_cfg   = load_operating_config()
config = dc.replace(_cfg, spec=dc.replace(_cfg.spec, c3h6_min_wtfrac=0.9945))  # BO と同じ純度緩和
BASE_PRICE = cp.LPG_C3H8_JPY_PER_KG

print(f"基準 LPG 単価 : C3H8 = {cp.LPG_C3H8_JPY_PER_KG} 円/kg, C4H10 = {cp.LPG_C4H10_JPY_PER_KG} 円/kg")
print(f"C3H6 出荷単価 : {cp.C3H6_PRODUCT_JPY_PER_KG} 円/kg")

## セル2: 最適点(BO ベスト)を読み込んで設計を組み立てる

`outputs/main_20260604_014318/best.json` に保存された BO ベスト trial の 23 変数を読み込み、
フローシート設計に変換する(`build_design` は main.py の `_build_design` と同一ロジックを移植。
optuna に依存しないよう、このノート内に再掲している)。


In [ ]:
# セル2: 最適点をロードして設計を構築
# --- main.py._build_design と同一ロジック(optuna 非依存にするためここに再掲)---
_FEED_STAGE_ABS = {"col2": (2, 9999), "col3": (70, 180)}

def _feed_stage_from_ratio(ratio, n, lo, hi):
    fs = int(round(ratio * n)); hi_eff = min(hi, n - 2); lo_eff = min(lo, hi_eff)
    return max(lo_eff, min(fs, hi_eff))

def build_design(p):
    n1 = int(p['col1_n_stages']); fs1 = int(p['col1_feed_stage'])
    n2 = int(p['col2_n_stages']); fs2 = _feed_stage_from_ratio(p['col2_feed_ratio'], n2, *_FEED_STAGE_ABS['col2'])
    n3 = int(p['col3_n_stages']); fs3 = _feed_stage_from_ratio(p['col3_feed_ratio'], n3, *_FEED_STAGE_ABS['col3'])
    p3 = float(p['col3_p_kpa'])
    reactor = CatofinDesignVars(T_in=p['T_in_K'], t_cyc=p['t_cyc_min'], D=p['D_reactor_m'],
                                L_bed=p['L_bed_m'], N_online=int(p['N_online']), d_p=float(p['d_p_mm']) / 1000.0)
    return FlowsheetDesignVars(
        swing=reactor,
        psa=PSADesignVars(D_col=p['D_psa_col_m'], L_bed=p['L_psa_bed_m'], desorption_target=p['desorption_target']),
        mem=MemDesignVars(P_H=p['P_H_Pa'], P_L=P_L_Pa, A_mem=p['A_mem_m2'], P_dist=p3 * 1000.0),
        dist1=ColumnTunables(P_col=float(p['col1_p_kpa']) * 1000.0, N_stages=n1, N_feed=1, reflux_ratio=2.0,
                             solver_method='sm', hysys_spec_value=float(p['col1_comp_frac_2']), hysys_feed_stage=fs1),
        dist2=ColumnTunables(P_col=float(p['col2_p_kpa']) * 1000.0, N_stages=n2, N_feed=1,
                             reflux_ratio=float(p['col2_reflux_ratio']), solver_method='hysys',
                             hysys_spec_value=float(p['col2_reflux_ratio']), hysys_feed_stage=fs2),
        dist3=ColumnTunables(P_col=p3 * 1000.0, N_stages=n3, N_feed=1, reflux_ratio=12.0,
                             solver_method='sm', hysys_spec_value=0.99, hysys_feed_stage=fs3),
    )

BEST = os.path.join(ROOT, 'outputs', 'main_20260604_014318', 'best.json')
with open(BEST, encoding='utf-8') as f:
    best = json.load(f)
params  = best['params']
design  = build_design(params)
F_fresh = float(params['F_C3H8_fresh_kmol_h'])

print(f"最適点      : trial #{best['number']}")
print(f"BO 記録 TAC : {best['effective_TAC']:.2f} 億円/年")
print(f"F_fresh     : {F_fresh:.1f} kmol/h")
print(f"反応器形式  : {REACTOR_KIND}(浅床軸流スイング)")

## セル3: 最適点を1回だけフル評価(← ここだけ HYSYS / VPN 必須)

リサイクルを含む全フローシートを収束させ、CAPEX・OPEX・Revenue・TAC・Profit を求める。
Dist2 は HYSYS で解くため、ここだけ数分かかり、VPN 接続が要る。
得られた物理結果 `one_pass` を、以降の価格スイープで**固定して使い回す**。


In [ ]:
# セル3: 最適点を1回だけフル評価
res = evaluate(design, config, verbose=False,
               apply_hi=APPLY_HI, hi_dT_min_K=HI_DT_MIN_K,
               apply_stage2=APPLY_STAGE2, F_C3H8_override=F_fresh)
assert res.economics_hi is not None, f"評価が feasible になりませんでした: {res.failure_reason}"

econ_hi  = res.economics_hi          # 熱統合(HI)後の経済(報告で使う基準)
one_pass = res.solver.one_pass       # 物理結果。価格スイープで固定して使い回す
prod_kmolh = one_pass['r3'].top.F_in.get('B', 0.0)   # C3H6 製品 [kmol/h]

print("評価成功(feasible = True)")
print(f"  CAPEX 年償却  : {econ_hi.total_capex / 8:9.2f} 億円/年")
print(f"  OPEX          : {econ_hi.total_opex:9.2f} 億円/年")
print(f"  TAC           : {econ_hi.TAC:9.2f} 億円/年")
print(f"  Revenue       : {econ_hi.total_revenue:9.2f} 億円/年")
print(f"  Profit        : {econ_hi.profit:9.2f} 億円/年  ({'黒字' if econ_hi.profit > 0 else '赤字'})")
print(f"  製造原単価     : {econ_hi.unit_jpy_per_t / 1000:9.2f} 円/kg")
print(f"  C3H6 生産量    : {prod_kmolh:.1f} kmol/h  (収率 {prod_kmolh / F_fresh * 100:.1f}%)")

## セル4: 「原料単価だけ差し替えて経済を再計算」する仕組み + 妥当性の実証

`econ_preHI_at(price)` は、固定した物理結果 `one_pass` に対して
**LPG 単価(C3H8・C4H10 とも `price`)だけを差し替えて**経済を計算し直す関数。

続けて2つの検証を実際に走らせる:

- **検証1**:価格を 1.5 倍にしたとき、値が変わる OPEX 項は
  「Fresh LPG 原料費」と「Hasebe 0.23·C_RM(原料費の間接費上乗せ)」の**2つだけ**であること。
  → 用役費・触媒・CAPEX などは1円も動かない=「原料費だけがスケールする」前提が正しい。
- **検証2**:TAC の変化分 ΔTAC が価格に対して**厳密に線形**(= 原料費総額 ×(倍率−1))であること。

> 補足:原料費は熱統合(HI)の影響を受けない項なので、ここで使う **HI 前**の ΔTAC は
> **HI 後**の ΔTAC と完全に一致する。だから「HI後 Profit」を基準に ΔTAC を足し引きしてよい。


In [ ]:
# セル4: 価格差し替えヘルパ + 妥当性の実証
MW_C3H6 = MW['B']

def econ_preHI_at(price):
    # one_pass を固定し、LPG 単価(C3H8/C4H10 とも price)だけ差し替えて経済(HI前)を再計算
    old3, old4 = econ_mod.LPG_C3H8_JPY_PER_KG, econ_mod.LPG_C4H10_JPY_PER_KG
    try:
        econ_mod.LPG_C3H8_JPY_PER_KG  = price
        econ_mod.LPG_C4H10_JPY_PER_KG = price
        return calculate_economics(one_pass, MW_C3H6)
    finally:
        econ_mod.LPG_C3H8_JPY_PER_KG, econ_mod.LPG_C4H10_JPY_PER_KG = old3, old4

econ_pre_base = econ_preHI_at(BASE_PRICE)

# --- 検証1: 価格 1.5 倍で変化する OPEX 項は「原料関連だけ」か ---
econ_pre_15 = econ_preHI_at(BASE_PRICE * 1.5)
changed = [(k, econ_pre_base.opex[k], econ_pre_15.opex.get(k))
           for k in econ_pre_base.opex
           if abs(econ_pre_base.opex[k] - econ_pre_15.opex.get(k, 0.0)) > 1e-9]
print("【検証1】価格を 1.5 倍にして変化した OPEX 項(原料関連だけのはず):")
for k, a, b in changed:
    print(f"    {k:36s} {a:8.2f} -> {b:8.2f} 億円/年")
n_total_opex = len(econ_pre_base.opex)
print(f"  → 変化した項は {len(changed)} / {n_total_opex} 項のみ。残り {n_total_opex - len(changed)} 項は不変。")

raw_total_base = sum(a for _, a, _ in changed)   # 原料関連 OPEX 総額(基準価格, 1.00+0.23倍込み)
print(f"\n原料関連 OPEX 総額(基準 {BASE_PRICE:.0f} 円/kg)= {raw_total_base:.2f} 億円/年"
      f"  … TAC の {raw_total_base / econ_hi.TAC * 100:.1f}% を占める")

# --- 検証2: ΔTAC は価格に対して厳密に線形か ---
print("\n【検証2】ΔTAC が価格倍率 f に対して線形か(実計算 vs 線形モデル):")
for f in (0.5, 0.7, 1.0, 1.3, 1.5):
    dTAC_real  = econ_preHI_at(BASE_PRICE * f).TAC - econ_pre_base.TAC
    dTAC_model = raw_total_base * (f - 1.0)
    print(f"    f={f:.1f}:  実計算 {dTAC_real:+8.2f}   線形モデル {dTAC_model:+8.2f}   "
          f"(差 {abs(dTAC_real - dTAC_model):.1e})")

## セル5: LPG 価格を振って Profit・損益分岐価格を計算

検証で「原料費だけが線形にスケールする」ことが確かめられたので、任意の価格での Profit は

$$ \text{Profit}(p) = \text{Revenue} - \Big[\text{TAC}_{95} + \text{原料費総額}\times(p/95 - 1)\Big] $$

で求まる(Revenue は価格に依存しない)。Profit は価格の**単調減少な直線**なので、
損益分岐価格(Profit = 0)は一意に決まる。**解析式**と**数値補間**の両方で出して突き合わせる。


In [ ]:
# セル5: 価格スイープ + 損益分岐価格
def metrics_at_price(price):
    dTAC   = econ_preHI_at(price).TAC - econ_pre_base.TAC   # 原料費の差(HI不変→HI後も同じ差)
    tac    = econ_hi.TAC + dTAC
    profit = econ_hi.total_revenue - tac
    unit   = tac * 1e8 / (econ_hi.annual_kg_C3H6 / 1000.0)  # 製造原単価 [円/ton]
    return profit, tac, unit

prices  = np.linspace(40, 140, 201)
profits = np.array([metrics_at_price(p)[0] for p in prices])
units   = np.array([metrics_at_price(p)[2] for p in prices]) / 1000.0   # [円/kg]

# 損益分岐価格(Profit = 0)
price_be = BASE_PRICE * (1.0 + econ_hi.profit / raw_total_base)        # 解析式
price_be_interp = float(np.interp(0.0, profits[::-1], prices[::-1]))   # 数値補間(クロスチェック)
sens = raw_total_base / BASE_PRICE                                     # 感度 [億円/年 per 円/kg]

summary = pd.DataFrame({
    '項目': ['基準 LPG 価格', '現行 Profit', '損益分岐 LPG 価格(解析)', '損益分岐 LPG 価格(数値)',
             '基準からの下落幅', '価格感度', '原料費が TAC に占める割合'],
    '値':  [f"{BASE_PRICE:.1f} 円/kg", f"{econ_hi.profit:.1f} 億円/年", f"{price_be:.1f} 円/kg",
            f"{price_be_interp:.1f} 円/kg", f"{(1 - price_be / BASE_PRICE) * 100:.1f} %",
            f"{sens:.2f} 億円/年 per 円/kg", f"{raw_total_base / econ_hi.TAC * 100:.1f} %"],
})
print(f"▶ LPG が約 {price_be:.1f} 円/kg 以下になると黒字化(現行 {BASE_PRICE:.0f} 円/kg では "
      f"{econ_hi.profit:.0f} 億円/年 の赤字)\n")
summary

## 図1: LPG 価格 vs 年間利益(損益分岐の本命グラフ)

レポート用に**単独の図**として保存する(PNG=プレビュー用、PDF=LaTeX 貼付用)。
黒字域は斜線ハッチング、赤字域はグレー塗りで区別(色に依存しない / rule.md §1.1)。


In [ ]:
# 図1: LPG 価格 vs 年間利益(グレースケール + ハッチング)
fig1, ax = plt.subplots(figsize=(7.0, 4.6))
ax.axhline(0, color='black', lw=0.8)

# 黒字域=斜線ハッチング, 赤字域=薄いグレー塗り(色に依存せず判別)
ax.fill_between(prices, profits, 0, where=(profits > 0),
                facecolor='none', hatch='////', edgecolor='0.45', linewidth=0.0)
ax.fill_between(prices, profits, 0, where=(profits < 0),
                facecolor='0.88', edgecolor='none')
ax.plot(prices, profits, lw=2.2, color='black')

# 損益分岐価格(破線)・現行価格(点線)
ax.axvline(price_be,   color='black', ls='--', lw=1.4)
ax.axvline(BASE_PRICE, color='black', ls=':',  lw=1.6)
ax.plot(BASE_PRICE, econ_hi.profit, 'o', color='black', ms=7, zorder=5)

# 注釈(すべて黒)
ax.annotate(f'損益分岐 {price_be:.1f} 円/kg', xy=(price_be, 0),
            xytext=(price_be + 11, max(profits) * 0.62), ha='left', color='black', fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black'))
ax.annotate(f'現行 {BASE_PRICE:.0f} 円/kg\n{econ_hi.profit:.0f} 億円/年',
            xy=(BASE_PRICE, econ_hi.profit), xytext=(BASE_PRICE + 7, econ_hi.profit - 110),
            ha='left', color='black', fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))
ax.text(46,  max(profits) * 0.26, '黒字', color='black', fontsize=13, fontweight='bold')
ax.text(123, min(profits) * 0.72, '赤字', color='black', fontsize=13, fontweight='bold')

ax.set_xlabel('LPG 原料単価 [円/kg]', fontsize=12)
ax.set_ylabel('年間利益 Profit [億円/年]', fontsize=12)
ax.set_title('LPG 原料価格と年間利益(最適設計を固定)', fontsize=12)
ax.set_xlim(prices[0], prices[-1])
fig1.tight_layout()
fig1.savefig('lpg_breakeven_profit.png', dpi=200, bbox_inches='tight')
fig1.savefig('lpg_breakeven_profit.pdf', bbox_inches='tight')
print("保存: monitor/lpg_breakeven_profit.png , .pdf")
plt.show()

## 図2: LPG 価格 vs C3H6 製造原単価(補足図)

製造原単価(TAC ÷ 年間生産量)が LPG 価格でどう動くかを示す補足図。破線は C3H6 の出荷単価。

> 注:図1の損益分岐(57.6 円/kg 付近)は H2 や燃料クレジットなど**全収入**を含めた利益ゼロ点。
> 一方この図で原単価が C3H6 出荷単価と交わる点は、副産物収入を無視した**より厳しい**目安であり、
> 両者は一致しない(本命は図1)。


In [ ]:
# 図2: LPG 価格 vs 製造原単価(グレースケール)
fig2, ax = plt.subplots(figsize=(7.0, 4.6))
ax.plot(prices, units, lw=2.2, color='black', label='C3H6 製造原単価(TAC 基準)')
ax.axhline(cp.C3H6_PRODUCT_JPY_PER_KG, color='black', ls='--', lw=1.4,
           label=f'C3H6 出荷単価 {cp.C3H6_PRODUCT_JPY_PER_KG:.0f} 円/kg')
ax.axvline(price_be,   color='0.5', ls='--', lw=1.2)
ax.axvline(BASE_PRICE, color='0.5', ls=':',  lw=1.4)

u_now = econ_hi.unit_jpy_per_t / 1000
ax.plot(BASE_PRICE, u_now, 'o', color='black', ms=6, zorder=5)
ax.annotate(f'現行 {u_now:.0f} 円/kg', xy=(BASE_PRICE, u_now),
            xytext=(BASE_PRICE + 5, u_now + 18), color='black', fontsize=10,
            arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlabel('LPG 原料単価 [円/kg]', fontsize=12)
ax.set_ylabel('C3H6 製造原単価 [円/kg]', fontsize=12)
ax.set_title('LPG 原料価格と C3H6 製造原単価(最適設計を固定)', fontsize=12)
ax.set_xlim(prices[0], prices[-1])
ax.legend(fontsize=10, loc='upper left', framealpha=1.0, edgecolor='0.3')
fig2.tight_layout()
fig2.savefig('lpg_breakeven_unitcost.png', dpi=200, bbox_inches='tight')
fig2.savefig('lpg_breakeven_unitcost.pdf', bbox_inches='tight')
print("保存: monitor/lpg_breakeven_unitcost.png , .pdf")
plt.show()

## まとめ

- 最適設計を固定して LPG 単価だけを振ると、**原料費 OPEX(1.00 + Hasebe 0.23 倍)だけが
  線形にスケール**し、用役費・触媒・CAPEX などは一切変わらない(セル4で実証)。
  そのため Profit は価格の**単調減少な直線**になる。
- **損益分岐 LPG 価格 ≒ 57.6 円/kg**。現行 95 円/kg より約 39% 安くなれば黒字化する。
  価格感度はおよそ **7.25 億円/年 per 1円/kg**(±10% で約 ±69 億円/年)。
- 図はレポート貼付用に2枚を別々のファイルへ保存(グリッドなし・四辺内向き目盛り・
  グレースケール+ハッチングで rule.md の図表ルールに準拠):
  - `monitor/lpg_breakeven_profit.png` / `.pdf` … 図1(本命:価格 vs 利益)
  - `monitor/lpg_breakeven_unitcost.png` / `.pdf` … 図2(補足:価格 vs 製造原単価)

### 注意(レポートに書くべき限界)

本分析は**設計を固定した単価感度**である。実際には LPG 価格が変われば BO の最適点自体も
動きうる。ただし目的関数は TAC 最小化であり、原料費が TAC の約 63% を占め、かつ F_fresh は
生産量バンド下限に制約律速されているため、価格変化に対する 23 変数の最適位置の移動は小さいと
予想される。厳密に確かめるには、各価格で main.py を再最適化して best を比較すればよい。
